# Rainflow counting

This page briefly explains how to perform a rainflow count on a time signal

In [ ]:
import numpy as np
import pandas as pd

import pylife.stress.rainflow as RF
import pylife.stress.timesignal as TS

import scipy.signal

import matplotlib.pyplot as plt


First let's quickly create a time signal.

In [ ]:
load_signal = TS.TimeSignalGenerator(
    10,
    {
        'number': 250,
        'amplitude_median': 50.0,
        'amplitude_std_dev': 0.5,
        'frequency_median': 4,
        'frequency_std_dev': 3,
        'offset_median': 0,
        'offset_std_dev': 0.4,
    },
).query(1_000_000)

Now let's get a [FourPointDetector](https://pylife.readthedocs.io/en/stable/stress/rainflow/fourpointdetector.html) to detect the hysteresis loops.

In [ ]:
detector = RF.FourPointDetector().process(load_signal)

Once the `load_signal` is processed we can get the load histogram from the recorder instance inside the detector class. Read more about rainflow detectors and recorders in the [rainflow overview page](https://pylife.readthedocs.io/en/stable/stress/rainflow.html).

Here we ask the recorder for a histogram with 64x64 bins..recorder.histogram(64)

In [ ]:
histogram = detector.recorder.histogram(64)

In [ ]:
from_levels, to_levels = histogram.index.levels
matrix = histogram.to_numpy().reshape(len(from_levels), len(to_levels))
fig, ax = plt.subplots()
ax.pcolormesh(from_levels.left, to_levels.left, matrix.T, cmap="jet")
ax.set(xlabel="From", ylabel="To")

You can use the `histogram` object to perform lifetime calculations and damage estimations. This is shown in [Life Time Calculation](../demos/lifetime_calc.ipynb)

In [ ]:
np.random.seed(4711)
sample_frequency = 4096
t = np.linspace(0, 60, 60 * sample_frequency)
white_noise = 80 * np.random.randn(len(t))
sine = 240 * np.sin(2 * np.pi * 50 * t)
signal = pd.DataFrame(
    {
        "white_noise": white_noise,
        "sine_over_noise": white_noise + sine,
    },
    index=pd.Index(t, name="time")
)

In [ ]:
histogram = RF.FKMDetector().process(signal["white_noise"]).recorder.histogram(64)

In [ ]:
from_levels, to_levels = histogram.index.levels
matrix = histogram.to_numpy().reshape(len(from_levels), len(to_levels))
fig, ax = plt.subplots()
ax.pcolormesh(matrix.T, cmap="jet")
ax.set(xlabel="From", ylabel="To")

In [ ]:
histogram = RF.FKMDetector().process(signal["sine_over_noise"]).recorder.histogram(64)

In [ ]:
from_levels, to_levels = histogram.index.levels
matrix = histogram.to_numpy().reshape(len(from_levels), len(to_levels))
fig, ax = plt.subplots()
ax.pcolormesh(matrix.T, cmap="jet")
ax.set(xlabel="From", ylabel="To")